# Token Economics and Artifacts

In production, tokenization is not just a preprocessing step; it's a **billing unit** and a **context constraint**. 

This notebook covers:
1. **Cost Estimation**: Predicting API costs before sending requests.
2. **Truncation Strategies**: Handling overflow without breaking semantics.
3. **Tokenizer Artifacts**: How special tokens (control tokens) affect your prompt length.
4. **Multilingual Efficiency**: Why some languages cost 3-5x more than English.

In [ ]:
# !pip install tiktoken transformers

## 1. Tiktoken (OpenAI)

OpenAI models use the `cl100k_base` encoding (for GPT-4 and GPT-3.5) or `o200k_base` (for GPT-4o).

In [ ]:
import tiktoken

def analyze_tokens(text: str, model: str = "gpt-4o"):
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)
    
    # 1. Basic Count
    print(f"Model: {model} | Encoding: {enc.name}")
    print(f"Token Count: {len(tokens)}")
    
    # 2. Industrial Artifacts: Byte-level inspection
    # Some tokens are 'glued' with spaces.
    for t in tokens[:10]:
        print(f"  ID: {t:<6} -> Raw: {enc.decode([t])!r}")

analyze_tokens("Tokenization is tricky.   Extra spaces   matter.")

## 2. Comparing Different Tokenizers

Let's compare how Llama 3 and GPT-4 handle the same text.

In [ ]:
from transformers import AutoTokenizer

text = "你好，世界！"

# GPT-4o (via tiktoken)
gpt_tokens = enc.encode(text)
print(f"GPT-4o tokens for '{text}': {len(gpt_tokens)}")

# Llama-3 (requires local tokenizer download or huggingface login)
# llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
# llama_tokens = llama_tokenizer.encode(text)
# print(f"Llama-3 tokens: {len(llama_tokens)}")

## 3. The 'Space' Sensitivity

Notice how leading spaces change the tokens.

In [ ]:
print(f"'apple': {enc.encode('apple')}")
print(f"' apple': {enc.encode(' apple')}")

## 4. Truncation & Cost Calculation

In production, we often truncate from the **middle** or **beginning** to keep the most recent context.

```python
def smart_truncate(text, max_tokens, model="gpt-4o"):
    enc = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)
    if len(tokens) <= max_tokens:
        return text
    
    # Truncate keeping the END (most relevant for chat)
    truncated_tokens = tokens[-max_tokens:]
    return enc.decode(truncated_tokens)

def estimate_cost(input_tokens, output_tokens, model="gpt-4o"):
    # Approx 2025 rates (USD per 1M tokens)
    rates = {
        "gpt-4o": {"in": 2.50, "out": 10.00},
        "gpt-4o-mini": {"in": 0.15, "out": 0.60}
    }
    cost = (input_tokens / 1e6 * rates[model]["in"]) + (output_tokens / 1e6 * rates[model]["out"])
    return f"${cost:.4f}"
```